# Gurobi nanečisto: malá pekárna
**Cvičení pro Jirku · KO · od jednoduchého modelu k souborům**

Tady trénuješ hlavně **překlad hotové matematické formulace do Gurobi**. Hlavní úloha je jednoduchá; potom následuje samostatný krátký bonus na dvouindexové proměnné. Není to skutečné zadání testu, ale procvičuje stejné základní dovednosti.

**Jednotný styl:** `import gurobipy as g`, název `model`, `addVars` pro kolekci proměnných, `addVar` pro jednu proměnnou, **vnější `for` + `addConstr`** pro omezení, `g.quicksum` pro součty. Vstup čteme tvým zápisem `[int(v) for v in ...]`.

### Jak pracovat
1. Vyber Python kernel s Gurobi a spusť importy.
2. Přečti zadání a matematický model.
3. V buňce **TVŮJ MODEL** nahraď `None` a doplň řádky `TODO`.
4. Spusť buňku s funkcí a potom kontroly. Po každé úpravě **znovu spusť definici funkce**.
5. Pak doplň načítání a zápis souborů a nakonec bonus.

Pracuj postupně, ne přes Run All nad nedoplněnou verzí. Úmyslné `NotImplementedError` znamená, že zbývá doplnit příslušnou část. Testovací buňky neupravuj. Hotové řešení modelu tu není.

**Procvičíš:** `INTEGER`, `BINARY`, `addVar`, `addVars`, `lb`, součty, cykly, Big-M s danou mezí, cíl, `optimize`, `Status`, `.X`, `.ObjVal`, vstup/výstup a v bonusu `x[i, j]`. Spojité proměnné ani pokročilé algoritmy tento trénink nepotřebuje.


In [1]:
import sys
from pathlib import Path
from itertools import product
from copy import deepcopy
import gurobipy as g

print("Python:", sys.executable)
print("Gurobi:", g.gurobi.version())

# Jen kontrola, ze lze zalozit model (vcetne licence).
check_model = g.Model("kontrola_prostredi")
check_model.dispose()
print("Import a zalozeni modelu: OK")


Python: /usr/bin/python3.10
Gurobi: (13, 0, 3)
Set parameter Username
Set parameter LicenseID to value 2866886
Academic license - for non-commercial use only - expires 2027-09-23
Import a zalozeni modelu: OK


Pokud import selže, vyber prostředí, ve kterém máš Gurobi nainstalované. Notebook běží přes **vybraný kernel**, který nemusí být stejný jako `python3` v terminálu. Chyba licence se řeší zvlášť; není to chyba tvých omezení. Tento notebook sám nic neinstaluje.


## 1. Zadání: co dnes upečeme?
Pekárna vyrábí celé kusy tří výrobků. Z tabulky vybíráš, kolik kusů každého upečeš.

| Index `i` | Výrobek | Příspěvek k zisku za kus | Čas na kus | Surovina na kus | Nejvýše kusů |
|---|---|---:|---:|---:|---:|
| 0 | Rohlík | 8 | 1 | 2 | 10 |
| 1 | Koláč | 13 | 3 | 2 | 8 |
| 2 | Muffin | 10 | 2 | 3 | 6 |

Máš **18 jednotek času** a **24 jednotek suroviny**. Celkem musíš upéct **alespoň 6 kusů**. Aby ses mohl pustit do koláčů, musíš zaplatit **jednorázový pronájem 12**. Bez pronájmu nesmíš vyrobit ani jeden koláč; zaplacený pronájem tě k výrobě nenutí.

**Maximalizuj součet příspěvků k zisku minus pronájem.** Neřešíme pořadí pečení ani konkrétní čas začátku.

### Formulace je hotová, tvým úkolem je naprogramovat ji
Data jsou známá čísla a seznamy. Jediné rozhodovací proměnné:
- $x_i\in\mathbb Z_{\geq0}$: počet kusů výrobku $i$;
- $y\in\{0,1\}$: zda zaplatíme pronájem.

$$\max\quad\sum_{i=0}^{n-1}p_i x_i-fy$$
$$\sum_i t_i x_i\leq T,\qquad \sum_i s_i x_i\leq S$$
$$\sum_i x_i\geq K,\qquad x_i\leq u_i\quad\text{pro každé }i$$
$$x_1\leq u_1y$$

V kódu: `price` = $p$, `time` = $t$, `material` = $s$, `fee` = $f$, `time_cap` = $T$, `material_cap` = $S$, `min_qty` = $K$, `limit` = $u$.

Poslední omezení je Big-M s **platnou mezí `limit[1]`**. Nemusíš žádné M vymýšlet. V testech jsou vždy právě tři výrobky a pronájem se týká indexu 1.


In [2]:
data = {
    "n": 3,
    "price": [8, 13, 10],
    "time": [1, 3, 2],
    "material": [2, 2, 3],
    "limit": [10, 8, 6],
    "time_cap": 18,
    "material_cap": 24,
    "min_qty": 6,
    "fee": 12,
}
print(data)


{'n': 3, 'price': [8, 13, 10], 'time': [1, 3, 2], 'material': [2, 2, 3], 'limit': [10, 8, 6], 'time_cap': 18, 'material_cap': 24, 'min_qty': 6, 'fee': 12}


## 2. TVŮJ MODEL
Funkce dostane data a vrátí `model, x, y`. Díky funkci můžeme stejný tvůj kód otestovat na několika vstupech. **Uvnitř používej `d`, nikoli globální `data`.**

Nejdřív vytvoř proměnné; potom omezení 1–5; potom cíl a `optimize()`. Na konci ponech `return`. Jedno omezení typu součet potřebuje jediný `addConstr`; omezení „pro každý výrobek“ potřebuje vnější cyklus. Dva přednastavené parametry nech být: vypínají dlouhý log a pomáhají získat jednoznačný stav nesplnitelnosti.


In [ ]:
def solve_bakery(d):
    n = d["n"]
    model = g.Model("pekarna")
    model.Params.OutputFlag = 0  # Pro vypis solveru zmen na 1.
    model.Params.DualReductions = 0  # Jednoznacny stav nesplnitelnosti.

    # TODO A: n nezapornych celociselnych promennych x[i].
    x = model.addVars(n, vtype=g.GRB.INTEGER, lb=0, name="x")

    # TODO B: jedna binarni promenna y.
    y = model.addVar(vtype=g.GRB.BINARY, name="y")

    if x is None or y is None:
        raise NotImplementedError("Dopln vytvoreni x a y.")

    # TODO 1: soucet spotrebovaneho casu <= time_cap.
    model.addConstr(
        g.quicksum(d["time"][i]* x[i] for i in range(n)) <= d["time_cap"]
    )

    # TODO 2: soucet spotrebovane suroviny <= material_cap.
    model.addConstr(
        g.quicksum(d["material"][i]* x[i] for i in range(n)) <= d["material_cap"]
    )
    # TODO 3: soucet vyrobenych kusu >= min_qty.
    model.addConstr(
        g.quicksum(x[i] for i in range(n)) >= d["min_qty"]
    )

    # TODO 4: pro kazde i plati x[i] <= limit[i].
    # Pouzij vnejsi for a uvnitr jeden addConstr.
    for i in range (n):
        model.addConstr(x[i] <= d["limit"][i])

    # TODO 5: kolace jen se zaplacenym pronajmem.
    model.addConstr(x[1] <= y*10000)

    # TODO 6: maximalizuj soucet price[i] * x[i] minus fee * y.
    model.setObjective(g.quicksum(d["price"][i] * x[i]for i in range(n)) - (d["fee"]*y), g.GRB.MAXIMIZE )
    

    # TODO 7: spust optimalizaci.
    model.optimize()

    return model, x, y


<details><summary><b>Nápověda k syntaxi – otevři, až ji potřebuješ</b></summary>

Tohle jsou obecné vzory, ne dosazené řešení:

```python
v = model.addVars(pocet, vtype=g.GRB.INTEGER, lb=0)
b = model.addVar(vtype=g.GRB.BINARY)
model.addConstr(g.quicksum(v[i] for i in range(pocet)) <= mez)
for i in range(pocet):
    model.addConstr(v[i] <= horni_meze[i])
model.setObjective(vyraz, g.GRB.MAXIMIZE)
model.optimize()
```

`g.quicksum(koef[i] * v[i] for i in range(pocet))` je vážený součet. `d["time"][i]` je známé číslo, `x[i]` je rozhodovací proměnná. Běžné Pythonové `if x[1] > 0` nepoužívej pro aktivaci pronájmu.
</details>


## 3. Kontrola modelu
Následující pomocnou buňku pouze spusť. Správné optimum hledá **nezávisle vyzkoušením všech malých kombinací**, ne druhým Gurobi modelem. Neporovnává konkrétní vektor, takže uzná i jiné optimální řešení.

Kontroluje typy a meze proměnných, stav řešiče, přípustnost nalezených hodnot a správnou hodnotu cíle. Testy nejsou matematický důkaz pro všechny vstupy, ale změny dat pomáhají odhalit vynechané podmínky i čísla zapsaná natvrdo.


In [ ]:
def oracle_bakery(d):
    best = None
    for quantities in product(*(range(v + 1) for v in d["limit"])):
        if sum(quantities) < d["min_qty"]:
            continue
        if sum(a*b for a,b in zip(quantities,d["time"])) > d["time_cap"]:
            continue
        if sum(a*b for a,b in zip(quantities,d["material"])) > d["material_cap"]:
            continue
        for rented in (0, 1):
            if quantities[1] > d["limit"][1] * rented:
                continue
            value = sum(a*b for a,b in zip(quantities,d["price"])) - d["fee"]*rented
            if best is None or value > best:
                best = value
    return best

def check_bakery(d, verbose=True):
    result = solve_bakery(deepcopy(d))
    assert isinstance(result, tuple) and len(result) == 3, "Vrat model, x, y."
    model, x, y = result
    try:
        model.update()
        assert all(x[i].VType == g.GRB.INTEGER for i in range(d["n"])), "x musi byt INTEGER."
        assert all(abs(x[i].LB) < 1e-8 for i in range(d["n"])), "x musi mit lb=0."
        assert y.VType == g.GRB.BINARY, "y musi byt BINARY."
        expected = oracle_bakery(d)
        if expected is None:
            assert model.Status == g.GRB.INFEASIBLE, f"Vstup nema reseni; stav je {model.Status}."
            if verbose: print("OK: spravne rozpoznana nesplnitelnost.")
            return
        assert model.Status == g.GRB.OPTIMAL, f"Ocekavam OPTIMAL; stav {model.Status}. Zavolal jsi optimize()?"
        q = [x[i].X for i in range(d["n"])]
        rented = y.X
        eps = 1e-5
        assert all(abs(v-round(v)) < eps for v in q), "Pocty nejsou cele."
        assert all(-eps <= q[i] <= d["limit"][i]+eps for i in range(d["n"])), "Porusena mez poctu."
        assert sum(q) >= d["min_qty"]-eps, "Chybi minimalni pocet kusu."
        assert sum(a*b for a,b in zip(q,d["time"])) <= d["time_cap"]+eps, "Prekrocen cas."
        assert sum(a*b for a,b in zip(q,d["material"])) <= d["material_cap"]+eps, "Prekrocena surovina."
        assert q[1] <= d["limit"][1]*rented+eps, "Kolace bez pronajmu."
        actual = sum(a*b for a,b in zip(q,d["price"])) - d["fee"]*rented
        assert abs(model.ObjVal-actual) < eps, "Cil neodpovida zisku minus pronajem."
        assert abs(actual-expected) < eps, f"Reseni neni optimalni: {actual}; optimum {expected}."
        if verbose: print("OK: pocty", [round(v) for v in q], "pronajem", round(rented), "zisk", round(actual))
    finally:
        model.dispose()


In [ ]:
# Nejdriv musi projit zakladni zadani.
check_bakery(data)


In [ ]:
# Pak zkus zmeny vstupu. Nic uvnitr solve_bakery nesmi byt natvrdo.
cases = [
    ("zaklad", {}),
    ("drahý pronajem", {"fee": 100}),
    ("pronajem zdarma", {"fee": 0}),
    ("malo casu", {"time_cap": 7}),
    ("malo surovin", {"material_cap": 13}),
    ("zmenene ceny", {"price": [2, 30, 1]}),
    ("zmenene spotreby", {"time": [4, 1, 3], "material": [1, 4, 2]}),
    ("nove limity", {"limit": [2, 2, 2]}),
    ("nesplnitelne minimum", {"min_qty": 25}),
    ("nula kusu dovolena", {"time_cap": 0, "material_cap": 0, "min_qty": 0}),
]
passed = 0
for label, changes in cases:
    current = deepcopy(data)
    current.update(changes)
    try:
        check_bakery(current, verbose=False)
        print("OK  ", label)
        passed += 1
    except Exception as e:
        print("CHYBA", label, "->", type(e).__name__, str(e))
print(f"Uspesne testy: {passed}/{len(cases)}")
assert passed == len(cases), "Oprav model a spust znovu definici funkce i testy."


## 4. Přečti řešení sám
Teď už nevypisuješ objekty `x[i]`, ale jejich hodnoty **`x[i].X`**. Doplň počty, pronájem a hodnotu cíle. Pokud model není vyřešený, `.X` nečti.


In [ ]:
model, x, y = solve_bakery(data)
if model.Status == g.GRB.OPTIMAL:
    # TODO: seznam celych poctu, binarni hodnota a celociselny cil.
    quantities = None
    rented = None
    profit = None
    if any(v is None for v in (quantities, rented, profit)):
        raise NotImplementedError("Dopln quantities, rented a profit.")
    assert quantities == [round(x[i].X) for i in range(data["n"])]
    assert rented == round(y.X)
    assert profit == round(model.ObjVal)
    print("Vyroba:", quantities, "Pronajem:", rented, "Zisk:", profit)
elif model.Status == g.GRB.INFEASIBLE:
    print("Zadani nema reseni.")
else:
    raise RuntimeError(f"Neocekavany stav {model.Status}")


## 5. Vstup ze souboru jako u praktického testu
V notebooku použijeme cestu `input_path`. Při odevzdání skriptu by vstupní cesta typicky přišla v `sys.argv[1]`. V notebooku na `sys.argv[1]` nespoléhej.

Formát této vymyšlené úlohy:
```text
n
price[0] price[1] price[2]
time[0] time[1] time[2]
material[0] material[1] material[2]
limit[0] limit[1] limit[2]
time_cap material_cap min_qty fee
```
Tato přípravná buňka vytvoří soubor v podadresáři pro cvičení. Pokud už tam existuje soubor jiného obsahu, nepřepíše ho.


In [ ]:
workdir = Path("gurobi_cviceni_data")
workdir.mkdir(exist_ok=True)
input_path = workdir / "pekarna_vstup.txt"
input_text = "3\n8 13 10\n1 3 2\n2 2 3\n10 8 6\n18 24 6 12\n"
if input_path.exists():
    assert input_path.read_text(encoding="utf-8") == input_text, "Soubor uz existuje s jinym obsahem. Zvol jinou cestu."
else:
    input_path.write_text(input_text, encoding="utf-8")
print(input_path.read_text(encoding="utf-8"))


In [ ]:
def load_bakery(path):
    with open(path, "r", encoding="utf-8") as f:
        # TODO: n je jedno cislo, dalsi 4 radky jsou seznamy.
        n = None
        price = None
        time = None
        material = None
        limit = None

        # TODO: posledni radek rozbal do 4 jednotlivych cisel.
        time_cap = material_cap = min_qty = fee = None

    if any(v is None for v in (n, price, time, material, limit,
                               time_cap, material_cap, min_qty, fee)):
        raise NotImplementedError("Dopln nacteni vstupu.")
    return dict(n=n, price=price, time=time, material=material,
                limit=limit, time_cap=time_cap, material_cap=material_cap,
                min_qty=min_qty, fee=fee)

loaded = load_bakery(input_path)
assert loaded == data, f"Chybne nacteno: {loaded}"
print("Vstup: OK")


<details><summary>Nápověda ke vstupu</summary>

Jedno číslo: `int(f.readline())`.

Seznam čísel: `[int(v) for v in f.readline().split()]`.

Čtyři čísla zvlášť: `a, b, c, d = [int(v) for v in f.readline().split()]`.

Každé `readline()` posune čtení na další řádek.
</details>


## 6. Zapiš výstup do souboru
Formát **této úlohy**:
1. řádek: celočíselný zisk;
2. řádek: počty tří výrobků oddělené mezerami;
3. řádek: 0 nebo 1 podle pronájmu.

Pokud je úloha **prokazatelně nesplnitelná**, zapiš pouze `-1`. Tento speciální význam stanovujeme jen pro toto cvičení. Jiný stav než OPTIMAL nebo INFEASIBLE nepovažuj automaticky za nesplnitelnost. Výstupní soubor s daným jménem se při `"w"` přepíše.


In [ ]:
def save_bakery(path, model, x, y, n):
    if model.Status == g.GRB.OPTIMAL:
        with open(path, "w", encoding="utf-8") as f:
            # TODO: zapis vsechny tri radky, za kazdy pridej "\n".
            raise NotImplementedError("Dopln zapis reseni.")
    elif model.Status == g.GRB.INFEASIBLE:
        with open(path, "w", encoding="utf-8") as f:
            # TODO: zapis jediny radek -1.
            raise NotImplementedError("Dopln zapis nesplnitelnosti.")
    else:
        raise RuntimeError(f"Neocekavany stav {model.Status}")


In [ ]:
output_path = workdir / "pekarna_vystup.txt"
model, x, y = solve_bakery(loaded)
save_bakery(output_path, model, x, y, loaded["n"])
lines = output_path.read_text(encoding="utf-8").splitlines()
assert len(lines) == 3, "Vystup musi mit tri radky."
assert int(lines[0]) == round(model.ObjVal)
assert [int(v) for v in lines[1].split()] == [round(x[i].X) for i in range(loaded["n"])]
assert int(lines[2]) == round(y.X)
model.dispose()
print("Zapis reseni: OK")
print(output_path.read_text(encoding="utf-8"))

impossible = deepcopy(loaded)
impossible["min_qty"] = 25
model, x, y = solve_bakery(impossible)
bad_path = workdir / "pekarna_nesplnitelna.txt"
save_bakery(bad_path, model, x, y, impossible["n"])
assert bad_path.read_text(encoding="utf-8").split() == ["-1"]
model.dispose()
print("Zapis nesplnitelnosti: OK")


<details><summary>Nápověda k výstupu</summary>

`round(x[i].X)` vrátí celočíselný počet, `round(model.ObjVal)` celočíselný cíl.

```python
f.write(str(cislo) + "\n")
f.write(" ".join(str(v) for v in hodnoty) + "\n")
```

Zde jsou hodnoty cíle celé, protože všechna vstupní čísla i proměnné jsou celočíselné. Obecný ILP může mít desetinné koeficienty a jeho cíl nemusí být celé číslo.
</details>


## 7. Krátký bonus: kdo zabalí objednávky?
Samostatná malá úloha na **dvouindexové `addVars` a dvojité cykly**. Výrobu z předchozí části už neměníme.

Máme 2 pracovníky a 3 objednávky. Každá objednávka musí dostat **právě jednoho pracovníka**. Pracovník 0 i pracovník 1 zvládne nejvýše 2 objednávky. Cena za přidělení je:

| Pracovník / objednávka | 0 | 1 | 2 |
|---|---:|---:|---:|
| 0 | 2 | 5 | 8 |
| 1 | 6 | 2 | 3 |

Pracovník 0 **nesmí dostat objednávku 2**. Minimalizuj cenu.

$a_{i,j}\in\{0,1\}$: pracovník $i$ dostal objednávku $j$.

$$\sum_i a_{i,j}=1\quad\forall j,\qquad
\sum_j a_{i,j}\leq cap_i\quad\forall i$$
$$a_{i,j}=0\quad\forall(i,j)\in forbidden$$
$$\min\quad\sum_i\sum_j cost_{i,j}a_{i,j}$$

Rozlišuj `cost[i][j]` (Pythonový seznam seznamů) a `a[i, j]` (Gurobi kolekce proměnných). Pro zákaz můžeš iterovat přímo `for i, j in forbidden:`. Přiřazení nemusí vypadat stejně jako referenční, pokud má stejnou optimální cenu.


In [ ]:
packing = {
    "workers": 2, "orders": 3,
    "cost": [[2, 5, 8], [6, 2, 3]],
    "cap": [2, 2],
    "forbidden": [(0, 2)],
}

def solve_packing(d):
    I, J = d["workers"], d["orders"]
    model = g.Model("baleni")
    model.Params.OutputFlag = 0
    model.Params.DualReductions = 0

    # TODO: tabulka I x J binarnich promennych.
    a = None
    if a is None:
        raise NotImplementedError("Dopln a pomoci addVars.")

    # TODO: pro kazdou objednavku soucet pres pracovniky == 1.

    # TODO: pro kazdeho pracovnika soucet objednavek <= jeho cap.

    # TODO: zakazane dvojice musi mit hodnotu 0.

    # TODO: minimalizace ceny (soucet pres oba indexy).

    # TODO: spust reseni.

    return model, a


In [ ]:
def check_packing(d):
    I, J = d["workers"], d["orders"]
    possible = []
    for owners in product(range(I), repeat=J):
        if any(owners.count(i) > d["cap"][i] for i in range(I)):
            continue
        if any((owners[j],j) in d["forbidden"] for j in range(J)):
            continue
        possible.append(sum(d["cost"][owners[j]][j] for j in range(J)))
    model, a = solve_packing(deepcopy(d))
    try:
        model.update()
        assert all(a[i,j].VType == g.GRB.BINARY for i in range(I) for j in range(J))
        if not possible:
            assert model.Status == g.GRB.INFEASIBLE, "Ocekavam nesplnitelnost."
            return
        assert model.Status == g.GRB.OPTIMAL, "Ocekavam OPTIMAL."
        chosen = [[round(a[i,j].X) for j in range(J)] for i in range(I)]
        assert all(sum(chosen[i][j] for i in range(I)) == 1 for j in range(J)), "Prirazeni objednavek."
        assert all(sum(chosen[i]) <= d["cap"][i] for i in range(I)), "Kapacity pracovniku."
        assert all(chosen[i][j] == 0 for i,j in d["forbidden"]), "Zakazane dvojice."
        value = sum(d["cost"][i][j]*chosen[i][j] for i in range(I) for j in range(J))
        assert abs(value-model.ObjVal) < 1e-5, "Nespravne zapsany cil."
        assert abs(value-min(possible)) < 1e-5, "Reseni neni optimalni."
    finally:
        model.dispose()

packing_cases = [
    ("zaklad", {}),
    ("zmenene ceny", {"cost": [[7,1,0],[1,8,9]]}),
    ("jiny zakaz", {"forbidden": [(1,0),(0,1)]}),
    ("nedostatecna kapacita", {"cap": [1,1]}),
]
for label, changes in packing_cases:
    current = deepcopy(packing)
    current.update(changes)
    check_packing(current)
    print("OK:", label)
print("Bonus: vsechny testy prosly.")


## 8. Závěrečná vlastní kontrola
- [ ] Umím vytvořit jednu proměnnou a řadu/tabulku proměnných.
- [ ] Rozlišuji `INTEGER` a `BINARY`; vím, že `lb=0` zakazuje záporné počty.
- [ ] Vím, kdy dát `for` ven z `addConstr` a kdy dovnitř `quicksum`.
- [ ] Umím zapsat vážený součet, kapacitu a dolní mez.
- [ ] Umím propojit počet kusů s binární proměnnou pomocí platného M.
- [ ] Umím přepnout `MAXIMIZE` na `MINIMIZE`.
- [ ] Volám `optimize()` a teprve potom čtu stav a hodnoty.
- [ ] Rozlišuji objekt `x[i]` a číselnou hodnotu `x[i].X`.
- [ ] Umím načíst jedno číslo, seznam a několik čísel do samostatných proměnných.
- [ ] Umím zapsat přesný formát výstupu a zvlášť nesplnitelnost.
- [ ] V bonusu rozlišuji součty přes řádek a přes sloupec.

**Až vše projde:** zakryj svůj model a napiš `solve_bakery` znovu pouze podle matematické formulace. Tím nejlépe poznáš, zda už ovládáš zápis samostatně.

Při převodu do `main.py` přidej na první řádek `#!/usr/bin/env python3`, importy a svoje funkce. Cesty vezmi z `sys.argv[1]` a `sys.argv[2]`. Do odevzdávaného programu nekopíruj oracle ani testy a nenechávej v něm automatické vytváření vstupního souboru.

*Vymyšlené cvičení inspirované typy dodaných zadání KO. Úspěch zde neznamená pokrytí všech možných úloh; AND/OR, indikátory a pokročilejší formulace zůstávají na další trénink.*
